# 05. 양자화 실험

**목표**
- FP16 / INT8 / INT4 양자화 후 성능/속도 비교
- 정확도 손실 5% 이내에서 가장 빠른 조합 선택

**실행 환경**: Kaggle (T4/P100)

In [ ]:
!rm -rf /kaggle/working/packages
!rm -rf /kaggle/working/korean-doc-understanding
!git clone https://github.com/shimtaehun/korean-doc-understanding.git

In [ ]:
import sys
sys.path.append("/kaggle/working/korean-doc-understanding")

In [ ]:
!pip install -q "transformers==4.47.0" "tokenizers==0.21.0" "jiwer" "bitsandbytes" "accelerate"

In [ ]:
# 셀 4 — flash_attn mock (Kaggle에 flash_attn 없으므로 필수)
import sys
import types
import importlib.machinery
import transformers.utils.import_utils

for mod_name in ['flash_attn', 'flash_attn.flash_attn_interface', 'flash_attn.bert_padding']:
    mock = types.ModuleType(mod_name)
    mock.__spec__ = importlib.machinery.ModuleSpec(mod_name, loader=None)
    sys.modules[mod_name] = mock

transformers.utils.import_utils.is_flash_attn_2_available = lambda: False
print("flash_attn mock 완료")

In [ ]:
import os, time, torch, wandb, yaml
from datasets import load_dataset
from torch.utils.data import DataLoader
from torch.amp import autocast
print("torch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
# Kaggle Secrets에서 WandB API 키 로드
from kaggle_secrets import UserSecretsClient

api_key = UserSecretsClient().get_secret("WANDB_API_KEY")
wandb.login(key=api_key, relogin=True)

# Colab이라면:
# from google.colab import userdata
# os.environ["WANDB_API_KEY"] = userdata.get('WANDB_API_KEY')
# wandb.login()

In [ ]:
with open("/kaggle/working/korean-doc-understanding/configs/train_config.yaml") as f:
    cfg = yaml.safe_load(f)

hf_dataset = load_dataset(cfg["data"]["hf_dataset"])
test_samples = list(hf_dataset["test"].select(range(20)))
print(f"테스트 샘플 수: {len(test_samples)}")

## 1. 공통 추론 함수

In [ ]:
import re
import json
import torch
from jiwer import cer
from torch.amp import autocast
from src.data.dataset import cord_to_target_sequence


def extract_fields_from_sequence(text: str) -> dict:
    """텍스트 시퀀스에서 태그 기반 필드를 딕셔너리로 추출."""
    fields = {}
    for match in re.finditer(r"<s_(\w+)>(.*?)</s_\1>", text, re.DOTALL):
        key, value = match.group(1), match.group(2).strip()
        if key in fields:
            if isinstance(fields[key], list):
                fields[key].append(value)
            else:
                fields[key] = [fields[key], value]
        else:
            fields[key] = value
    return fields


def compute_field_f1(pred_fields: dict, gt_fields: dict) -> float:
    """예측 필드와 GT 필드 간 F1 스코어 계산."""
    all_keys = set(pred_fields.keys()) | set(gt_fields.keys())
    if not all_keys:
        return 0.0
    tp = sum(
        1 for k in all_keys
        if pred_fields.get(k) == gt_fields.get(k) and k in pred_fields and k in gt_fields
    )
    precision = tp / len(pred_fields) if pred_fields else 0.0
    recall = tp / len(gt_fields) if gt_fields else 0.0
    if precision + recall == 0:
        return 0.0
    return 2 * precision * recall / (precision + recall)


def benchmark_model(
    model,
    processor,
    test_samples: list,
    device: str,
    label: str = "FP16",
    max_new_tokens: int = 512,
    num_beams: int = 3,
) -> dict:
    """모델 추론 성능 벤치마크.

    Args:
        model: HuggingFace 모델.
        processor: 대응하는 프로세서.
        test_samples: 테스트 샘플 리스트.
        device: "cuda" 또는 "cpu".
        label: 결과 레이블 (예: "FP16-base", "INT8-base").
        max_new_tokens: 생성 최대 토큰 수.
        num_beams: 빔 서치 크기.

    Returns:
        벤치마크 결과 딕셔너리: label, field_f1, cer, avg_time_ms, model_size_mb.
    """
    PROMPT = "<DocVQA>"
    f1_scores = []
    cer_scores = []
    times_ms = []

    model.eval()

    for i, sample in enumerate(test_samples):
        try:
            gt_raw = json.loads(sample["ground_truth"])
            gt_sequence = cord_to_target_sequence(gt_raw)
            gt_fields = extract_fields_from_sequence(gt_sequence)

            image = sample["image"].convert("RGB")
            inputs = processor(
                text=PROMPT,
                images=image,
                return_tensors="pt",
            )
            # device_map="auto" 모델은 .to() 불필요, 그 외는 device로 이동
            if hasattr(model, "hf_device_map"):
                inputs = {k: v.to(next(model.parameters()).device) for k, v in inputs.items()}
            else:
                inputs = {k: v.to(device) for k, v in inputs.items()}

            t_start = time.time()
            with torch.no_grad():
                with autocast("cuda", dtype=torch.float16):
                    output_ids = model.generate(
                        input_ids=inputs["input_ids"],
                        pixel_values=inputs["pixel_values"],
                        max_new_tokens=max_new_tokens,
                        num_beams=num_beams,
                    )
            t_end = time.time()
            elapsed_ms = (t_end - t_start) * 1000.0

            generated = output_ids[:, inputs["input_ids"].shape[1]:]
            pred_sequence = processor.batch_decode(generated, skip_special_tokens=False)[0]
            pred_fields = extract_fields_from_sequence(pred_sequence)

            field_f1 = compute_field_f1(pred_fields, gt_fields)
            sample_cer = cer(gt_sequence, pred_sequence) if gt_sequence else 1.0

            f1_scores.append(field_f1)
            cer_scores.append(sample_cer)
            times_ms.append(elapsed_ms)

            print(f"  [{label}] [{i:02d}] F1={field_f1:.3f} | CER={sample_cer:.3f} | {elapsed_ms:.0f}ms")

        except Exception as e:
            print(f"  [{label}] [{i:02d}] 오류 발생: {e}")
            f1_scores.append(0.0)
            cer_scores.append(1.0)
            times_ms.append(float("nan"))

    # 유효한 시간 값만 평균 계산
    valid_times = [t for t in times_ms if t == t]  # NaN 제거
    avg_time_ms = sum(valid_times) / len(valid_times) if valid_times else float("nan")

    model_size_mb = sum(
        p.nelement() * p.element_size() for p in model.parameters()
    ) / 1024 / 1024

    result = {
        "label": label,
        "field_f1": sum(f1_scores) / len(f1_scores) if f1_scores else 0.0,
        "cer": sum(cer_scores) / len(cer_scores) if cer_scores else 1.0,
        "avg_time_ms": avg_time_ms,
        "model_size_mb": model_size_mb,
    }
    print(f"\n[{label}] 평균 → F1={result['field_f1']:.4f} | CER={result['cer']:.4f} | "
          f"{result['avg_time_ms']:.1f}ms/sample | {result['model_size_mb']:.1f}MB\n")
    return result

## 2. FP16 (베이스라인 - LoRA 미적용)

In [ ]:
from transformers import AutoModelForCausalLM, AutoProcessor

MODEL_ID = cfg["model"]["name"]
processor_fp16 = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
    torch_dtype=torch.float16,
).to(DEVICE)
model_fp16.eval()

result_fp16 = benchmark_model(model_fp16, processor_fp16, test_samples, DEVICE, label="FP16-base")
print(result_fp16)

del model_fp16, processor_fp16
torch.cuda.empty_cache()

## 3. LoRA 체크포인트 로드 + FP16

In [ ]:
CKPT_DIR = "/kaggle/working/checkpoints/best_model"

if os.path.exists(CKPT_DIR):
    from src.model.florence_lora import LoRASettings, load_florence_for_inference

    lora_settings = LoRASettings(
        r=cfg["lora"]["r"],
        alpha=cfg["lora"]["alpha"],
        dropout=cfg["lora"]["dropout"],
        target_modules=cfg["lora"]["target_modules"],
    )
    model_lora, processor_lora = load_florence_for_inference(
        model_id=MODEL_ID,
        checkpoint_dir=CKPT_DIR,
        device=DEVICE,
    )
    model_lora.eval()

    result_lora_fp16 = benchmark_model(model_lora, processor_lora, test_samples, DEVICE, label="LoRA-FP16")
    print(result_lora_fp16)

    del model_lora
    torch.cuda.empty_cache()
else:
    print(f"체크포인트 없음: {CKPT_DIR}")
    print("→ 03_train_lora.ipynb 먼저 실행 후 체크포인트 경로 확인")
    result_lora_fp16 = None

## 4. INT8 양자화 (bitsandbytes)

In [ ]:
from transformers import BitsAndBytesConfig

try:
    bnb_int8_config = BitsAndBytesConfig(load_in_8bit=True)

    model_int8 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_int8_config,
        device_map="auto",
    )
    processor_int8 = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model_int8.eval()

    result_int8 = benchmark_model(model_int8, processor_int8, test_samples, DEVICE, label="INT8-base")
    print(result_int8)

    del model_int8, processor_int8
    torch.cuda.empty_cache()
except Exception as e:
    print(f"INT8 양자화 로드 실패 (P100에서 발생 가능): {e}")
    result_int8 = None

## 5. INT4 양자화 (bitsandbytes NF4)

In [ ]:
try:
    bnb_int4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    model_int4 = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        trust_remote_code=True,
        quantization_config=bnb_int4_config,
        device_map="auto",
    )
    processor_int4 = AutoProcessor.from_pretrained(MODEL_ID, trust_remote_code=True)
    model_int4.eval()

    result_int4 = benchmark_model(model_int4, processor_int4, test_samples, DEVICE, label="INT4-NF4-base")
    print(result_int4)

    del model_int4, processor_int4
    torch.cuda.empty_cache()
except Exception as e:
    print(f"INT4 양자화 로드 실패 (P100에서 발생 가능): {e}")
    result_int4 = None

## 6. 결과 비교

In [ ]:
import pandas as pd

all_results = [r for r in [result_fp16, result_lora_fp16, result_int8, result_int4] if r is not None]
df = pd.DataFrame(all_results)
df["speedup"] = df["avg_time_ms"].iloc[0] / df["avg_time_ms"]
print(df.to_string(index=False))

wandb.init(
    project=cfg["wandb"]["project"],
    entity="sthun0211-home",
    name="quantization-benchmark",
    config={"num_samples": len(test_samples)},
)
wandb.log({"quantization_results": wandb.Table(dataframe=df)})
wandb.finish()

print("""
=== docs/EXPERIMENTS.md에 기록할 내용 ===
""")
print(df[["label", "field_f1", "cer", "avg_time_ms", "model_size_mb", "speedup"]].to_string(index=False))